# Airbnb Price Intelligence for NYC Listings
I had done a project for AMS 317 (Linear Regression Analysis) where my group and I looked into 2019 AirBnb dataset for New York City. We had done a complete data analysis, EDA, used ANOVA to measure the question "{Question}". 
I've always wanted to do a similar analysis on Airbnb dataset but better. Instead of just conducting an analysis. I wanted to build a model and maybe answer some business question. I am also a frequent Airbnb user when I go on trips and lots of times, I found myself asking, "Is this price reasonable in comparision to the surrounding Airbnb based on similar metrics and characteristics?" So I decided to look into the 2026 NYC Airbnb Dataset, and decided to explore the following question with real world messy data instead of a clean dataset.

# Business question
How can we estimate a fair nightly price for a New York City Airbnb listing using listing characteristics, and identify whether a host is pricing above or below market?

# Understanding the problem

Before even examining the data, I like to understand the problem first. 
* `What are we predicting?` : 
* `What are the metrics?`: 

## Load the dataset

This notebook loads the CSV directly from the project data folder using a simple relative path from the `notebooks/` directory.

In [41]:
import pandas as pd
import numpy as np

listings = pd.read_csv("../data/listings.csv")

print(f"Shape: {listings.shape[0]:,} rows x {listings.shape[1]} columns")
listings.info()

Shape: 36,353 rows x 18 columns
<class 'pandas.DataFrame'>
RangeIndex: 36353 entries, 0 to 36352
Data columns (total 18 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              36353 non-null  int64  
 1   name                            36351 non-null  str    
 2   host_id                         36353 non-null  int64  
 3   host_name                       34896 non-null  str    
 4   neighbourhood_group             36353 non-null  str    
 5   neighbourhood                   36353 non-null  str    
 6   latitude                        36353 non-null  float64
 7   longitude                       36353 non-null  float64
 8   room_type                       36353 non-null  str    
 9   price                           21415 non-null  float64
 10  minimum_nights                  36353 non-null  int64  
 11  number_of_reviews               36353 non-null  int64  
 12  last_review

In [42]:
listings.head()

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365,number_of_reviews_ltm,license
0,1274691077561855573,Beautiful room for rent hosted by Svitlana,658783772,Svitlana,Staten Island,Eltingville,40.535470,-74.151350,Private room,87.0,2,17,2025-10-15,1.62,1,73,17,OSE-STRREG-0002773
1,1274722590671904755,Exquisite & Charming Studio in Prime Locations,458668555,Patricia,Brooklyn,East Flatbush,40.651855,-73.945082,Private room,80.0,30,2,2025-06-30,0.21,6,93,2,NaN
2,1274773188072371454,"Private 2 Bedroom Unit, Full Kitchen, Washer/D...",22369387,Waleed,Staten Island,Silver Lake,40.620370,-74.101954,Entire home/apt,99.0,30,0,NaN,NaN,3,201,0,NaN
3,1274820999428562667,"Beautiful, Serene West Village Treehouse",505544503,Michael,Manhattan,West Village,40.735566,-74.004233,Entire home/apt,312.0,30,0,NaN,NaN,1,317,0,NaN
4,1274826163511555978,Renovated 1BR w/ Spa Shower,493151605,Kareem,Queens,Jamaica,40.704184,-73.783945,Entire home/apt,111.0,30,1,2025-01-03,0.10,4,333,1,NaN


# Drop Columns that are not useful for pricing model
Columns to drop: id, name, host_id, host_name, license, last_review, calculated_host_listings_count

In [43]:
listings_model = listings.drop(columns=[
    "id",
    "name",
    "host_id",
    "host_name",
    "license",
    "last_review"
])
listings_model.info()
listings_model.shape

<class 'pandas.DataFrame'>
RangeIndex: 36353 entries, 0 to 36352
Data columns (total 12 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   neighbourhood_group             36353 non-null  str    
 1   neighbourhood                   36353 non-null  str    
 2   latitude                        36353 non-null  float64
 3   longitude                       36353 non-null  float64
 4   room_type                       36353 non-null  str    
 5   price                           21415 non-null  float64
 6   minimum_nights                  36353 non-null  int64  
 7   number_of_reviews               36353 non-null  int64  
 8   reviews_per_month               25007 non-null  float64
 9   calculated_host_listings_count  36353 non-null  int64  
 10  availability_365                36353 non-null  int64  
 11  number_of_reviews_ltm           36353 non-null  int64  
dtypes: float64(4), int64(5), str(3)
memory usag

(36353, 12)

## Missing values

Missing-value checks help identify which columns may need later treatment, documentation, or exclusion decisions.

In [44]:
missing_counts = listings_model.isnull().sum()
missing_counts[missing_counts > 0]

price                14938
reviews_per_month    11346
dtype: int64

# Dropping Non-impact features
Lets drop the null price values so our modeling can use real accurate prices because price is the target.
Also, null values of reviews_per_month often means the listing probably is new, does not have any recent review for the year or not enough review history (it is only April of 2026)

* `id`: Unique listing identifier, not a meaningful pricing signal.
* `name`: Free-text title, too unstructured for a simple baseline pricing model.
* `host_id`: Unique host identifier, mostly acts like a label rather than a real feature.
* `host_name`: Host name is not meaningfully related to market price.
* `license`: Too many missing values, so it is weak for the first baseline model.
* `last_review`: Overall stronger review indicators in the dataset such as number_of_reviews and reviews_per_month, so safe to drop.


In [45]:
listings_model = listings_model.dropna(subset=["price"]).copy()
listings_model["reviews_per_month"] = listings_model["reviews_per_month"].fillna(0)
listings_model.isnull().sum()

neighbourhood_group               0
neighbourhood                     0
latitude                          0
longitude                         0
room_type                         0
price                             0
minimum_nights                    0
number_of_reviews                 0
reviews_per_month                 0
calculated_host_listings_count    0
availability_365                  0
number_of_reviews_ltm             0
dtype: int64

We have confirmed that the dataset has no more null values!

In [48]:
listings_model.head(10)

,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,reviews_per_month,calculated_host_listings_count,availability_365,number_of_reviews_ltm
0,Staten Island,Eltingville,40.535470,-74.151350,Private room,87.0,2,17,1.62,1,73,17
1,Brooklyn,East Flatbush,40.651855,-73.945082,Private room,80.0,30,2,0.21,6,93,2
2,Staten Island,Silver Lake,40.620370,-74.101954,Entire home/apt,99.0,30,0,0.00,3,201,0
3,Manhattan,West Village,40.735566,-74.004233,Entire home/apt,312.0,30,0,0.00,1,317,0
4,Queens,Jamaica,40.704184,-73.783945,Entire home/apt,111.0,30,1,0.10,4,333,1
5,Brooklyn,Sea Gate,40.578358,-74.004845,Entire home/apt,142.0,30,4,0.55,1,345,4
6,Queens,Ozone Park,40.675170,-73.859390,Private room,51.0,30,2,0.20,26,154,2
7,Brooklyn,South Slope,40.662972,-73.985403,Private room,50.0,30,4,0.37,3,266,4
8,Brooklyn,Williamsburg,40.713934,-73.944217,Entire home/apt,397.0,30,0,0.00,1,137,0
10,Manhattan,Inwood,40.865297,-73.924800,Entire home/apt,105.0,30,3,0.28,5,320,3
